In [1]:
import math, random, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
import lightgbm as lgb
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

PyTorch 2.10.0+cu128 | Device: cuda


In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')
print(train['class'].value_counts())

Train: (577347, 12) | Test: (247435, 11)
class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64


In [3]:
ID, TARGET = 'id', 'class'

train[TARGET] = train[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
X       = train.drop([ID, TARGET], axis=1)
y       = train[TARGET]
X_test  = test.drop([ID], axis=1)
test_id = test[ID]

_mask = (y == 2) & (X['redshift'] < 0.15)
_ug = X.loc[_mask, 'u'] - X.loc[_mask, 'g']
_gr = X.loc[_mask, 'g'] - X.loc[_mask, 'r']
LOCUS_A, LOCUS_B = np.polyfit(_ug, _gr, deg=1)
print(f'Stellar locus: g_r = {LOCUS_A:.4f}*u_g + {LOCUS_B:.4f}')

del train, test

def add_physics_features(df):
    d = df.copy()
    mags = ['u','g','r','i','z']

    # Color indices
    for a, b in [('u','g'),('g','r'),('r','i'),('i','z'),
                 ('g','z'),('u','r'),('u','z'),('g','i'),('r','z')]:
        d[f'{a}_{b}'] = d[a] - d[b]

    # Magnitude stats
    d['mag_mean']  = d[mags].mean(axis=1)
    d['mag_std']   = d[mags].std(axis=1)
    d['mag_range'] = d[mags].max(axis=1) - d[mags].min(axis=1)

    # NEW: spectral slope & curvature
    d['mag_slope']      = (-2*d['u'] - d['g'] + d['i'] + 2*d['z']) / 10
    d['mag_curvature']  = d['u'] - 2*d['r'] + d['z']
    d['blue_curvature'] = d['u'] - 2*d['g'] + d['r']
    d['red_curvature']  = d['r'] - 2*d['i'] + d['z']

    # Redshift transforms
    d['log_redshift'] = np.log1p(d['redshift'].clip(lower=0))
    d['redshift_sq']  = d['redshift'] ** 2
    d['redshift_zone'] = pd.cut(d['redshift'],
        bins=[-np.inf,0.0,0.15,0.50,1.0,1.3,np.inf], labels=[0,1,2,3,4,5]).astype(float)

    # NEW: redshift × magnitude & magnitude/redshift (top XGB features!)
    for b in mags:
        d[f'redshift_{b}'] = d['redshift'] * d[b]
        d[f'{b}_over_redshift'] = d[b] / (d['redshift'].abs() + 1e-6)

    # NEW: flux features
    for b in mags:
        d[f'flux_{b}'] = np.power(10.0, -0.4 * d[b].clip(-30, 30))
    flux_cols = [f'flux_{b}' for b in mags]
    d['flux_mean']  = d[flux_cols].mean(axis=1)
    d['flux_std']   = d[flux_cols].std(axis=1)
    d['flux_range'] = d[flux_cols].max(axis=1) - d[flux_cols].min(axis=1)

    # NEW: color-plane geometry
    d['color_plane_radius_ug_gr'] = np.sqrt(d['u_g']**2 + d['g_r']**2)
    d['color_plane_angle_ug_gr']  = np.arctan2(d['u_g'], d['g_r'] + 1e-6)
    d['color_plane_radius_ri_iz'] = np.sqrt(d['r_i']**2 + d['i_z']**2)
    d['color_plane_angle_ri_iz']  = np.arctan2(d['r_i'], d['i_z'] + 1e-6)

    # Sky coords
    d['sin_alpha'] = np.sin(np.radians(d['alpha']))
    d['cos_alpha'] = np.cos(np.radians(d['alpha']))
    d['sin_delta'] = np.sin(np.radians(d['delta']))
    d['cos_delta'] = np.cos(np.radians(d['delta']))

    # Physical 3D position
    r_dist = d['log_redshift']
    d['phys_x'] = r_dist * d['cos_delta'] * d['cos_alpha']
    d['phys_y'] = r_dist * d['cos_delta'] * d['sin_alpha']
    d['phys_z'] = r_dist * d['sin_delta']

    # Interactions
    d['redshift_x_gr'] = d['redshift'] * d['g_r']
    d['redshift_x_ur'] = d['redshift'] * d['u_r']

    # Stellar locus
    d['locus_gr_pred']  = LOCUS_A * d['u_g'] + LOCUS_B
    d['locus_distance'] = d['g_r'] - d['locus_gr_pred']
    d['locus_dist_abs'] = d['locus_distance'].abs()

    # Categorical encodings
    spec_map = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
    pop_map  = {'Blue_Cloud': 0, 'Red_Sequence': 1}
    d['spectral_enc'] = d['spectral_type'].map(spec_map).astype(float)
    d['pop_enc']      = d['galaxy_population'].map(pop_map).astype(float)
    d['spec_x_pop']   = d['spectral_enc'] * d['pop_enc']

    return d

X      = add_physics_features(X)
X_test = add_physics_features(X_test)
print(f'After physics features: X={X.shape}')

Stellar locus: g_r = 0.1474*u_g + 0.3133
After physics features: X=(577347, 66)


In [4]:
RAW_NUM_COLS = ['alpha','delta','u','g','r','i','z','redshift']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in RAW_NUM_COLS if c in X.columns]
category_map = {}

important_combos = sorted([
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
    ('spectral_type', 'galaxy_population'),
    ('redshift_cat_', 'g_cat_'),
    ('redshift_zone',),
])

def feature_engineering(df, fit=False):
    df = df.copy()
    df['_g_/_redshift'] = (df['g'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_r_/_redshift'] = (df['r'] / (df['redshift'] + 1e-6)).astype('float32')

    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            code_map = {cat: i for i, cat in enumerate(category_map[col])}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    for col in num_cols:
        cat_name = f'{col}_cat_'
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            code_map = {cat: i for i, cat in enumerate(category_map[col])}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    bin_config = {
        'delta':    [100, 500],
        'redshift': [50, 200],
        'g_r':      [20, 100],
        'u_r':      [20, 100],
    }
    for col, bins_list in bin_config.items():
        if col not in df.columns: continue
        for n_bins in bins_list:
            bin_name = f'{col}_{n_bins}_quantile_bin_'
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal',
                                      strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                category_map[bin_name] = kb
            else:
                binned = category_map[bin_name].transform(df[[col]]).ravel().astype('int32')
            df[bin_name] = binned
            df[bin_name] = df[bin_name].astype('category')

    combo_names = []
    for cols in important_combos:
        if not all(c in df.columns for c in cols): continue
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            code_map = {cat: i for i, cat in enumerate(category_map[combo_name])}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [c for c in df.columns if c.endswith('_')]
    new_num_cols = [c for c in df.columns if c.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)

cat_cols = sorted(cat_cols + new_cat_cols)
num_cols = num_cols + new_num_cols
X      = X.reindex(sorted(X.columns), axis=1)
X_test = X_test.reindex(sorted(X_test.columns), axis=1)

print(f'cat_cols: {len(cat_cols)} | num_cols: {len(num_cols)}')
print(f'X shape: {X.shape}')

cat_cols: 23 | num_cols: 11
X shape: (577347, 90)


In [5]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in
                      ('median_center','robust_scale','smooth_clip','l2_normalize')]
    def fit(self, X, y=None):
        if 'median_center' in self._tfms or 'robust_scale' in self._tfms:
            self._median = np.median(X, axis=0)
            q_diff = np.quantile(X,0.75,axis=0) - np.quantile(X,0.25,axis=0)
            zero_idx = q_diff == 0.0
            q_diff[zero_idx] = 0.5*(X.max(axis=0)[zero_idx]-X.min(axis=0)[zero_idx])
            self._iqr_factors = 1.0/(q_diff+1e-30)
            self._iqr_factors[q_diff==0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for tfm in self._tfms:
            if tfm=='median_center':  X -= self._median
            elif tfm=='robust_scale': X *= self._iqr_factors
            elif tfm=='smooth_clip':  X = X/np.sqrt(1+(X/3)**2)
            elif tfm=='l2_normalize':
                norms = np.linalg.norm(X,axis=1,keepdims=True)
                X /= np.where(norms==0,1.0,norms)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims
        self.onehot_features=[]; self.embed_layers=nn.ModuleList()
        self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        batch,n_ens,_=x.shape; features=[]
        if self.onehot_features:
            oh_x=x[:,:,self.onehot_features]
            oh_dims=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(batch,n_ens,sum(oh_dims),device=x.device)
            start=0
            for idx,dim in enumerate(oh_dims):
                enc.scatter_(2,oh_x[:,:,idx:idx+1].long()+start,1.0); start+=dim
            features.append(enc)
        for emb_list,fi in zip(self.embed_layers,self._embed_feature_indices):
            features.append(torch.cat([emb_list[k](x[:,k,fi:fi+1].long()) for k in range(n_ens)],dim=1))
        return torch.cat(features,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__()
        self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_features,out_features,bias=True):
        super().__init__()
        self.in_features=in_features; self.out_features=out_features
        self.weight=nn.Parameter(torch.randn(n_ens,in_features,out_features))
        self.bias=nn.Parameter(torch.randn(n_ens,out_features)) if bias else None
    def forward(self,x):
        x=torch.einsum('bki,kio->bko',x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__()
        self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.zeros(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        periodic=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1+self.b1))
        transformed=self.act(torch.einsum('bkfh,kfhd->bkfd',periodic,self.w2)+self.b2)
        return torch.cat([x.unsqueeze(-1),transformed],dim=-1).flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__()
        n_ens=cfg['n_ens']; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,cfg['embed_dim'],cfg['onehot_thresh'])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg['pbld_hidden_dim'],
                                     cfg['pbld_out_dim'],cfg['pbld_freq_scale'],cfg['pbld_activation'])
        num_emb_dim=n_numerical*cfg['pbld_out_dim']
        cat_emb_dim=sum(c if c<=cfg['onehot_thresh'] else cfg['embed_dim'] for c in cat_dims)
        total_dim=num_emb_dim+cat_emb_dim
        layers=[]
        if cfg['add_front_scale']: layers.append(ScalingLayer(n_ens,total_dim))
        self._dropout_modules=[]; in_dim=total_dim
        for i,h in enumerate(cfg['hidden_dims']):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            drop=nn.Dropout(cfg['dropout']); self._dropout_modules.append(drop)
            layers+=[lin,cfg['activation'](),drop]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x=torch.cat([self.num_embed(x_num),self.cate(x_cat)],dim=2)
        return F.softmax(self.output_layer(self.hidden(x)),dim=2)

print('Model components defined ✅')

Model components defined ✅


In [6]:
def apply_schedule(init_val,progress,sched,flat_ratio=0.3):
    if sched=='constant':    return init_val
    elif sched=='cos':       return init_val*(math.cos(math.pi*progress)+1)/2
    elif sched=='flat_cos':
        if progress<flat_ratio: return init_val
        t=(progress-flat_ratio)/(1-flat_ratio)
        return init_val*(math.cos(math.pi*t)+1)/2
    elif sched=='sqrt_cos':  return init_val*math.sqrt((math.cos(math.pi*progress)+1)/2)
    elif sched=='expm4t':    return init_val*math.exp(-4*progress)
    else: raise ValueError(f'Unknown: {sched}')

def get_parameter_groups(model,p):
    first_w_id=id(model.first_linear.weight)
    scale_p,pbld_p,first_w_p,other_w_p,bias_p=[],[],[],[],[]
    for name,param in model.named_parameters():
        if 'num_embed' in name: pbld_p.append(param)
        elif 'scale' in name:   scale_p.append(param)
        elif id(param)==first_w_id: first_w_p.append(param)
        elif 'bias' in name:    bias_p.append(param)
        else:                   other_w_p.append(param)
    LR,WD=p['lr'],p['weight_decay']
    return [
        {'params':scale_p,   'lr':LR*p['lr_scale_mult'],         'weight_decay':WD*p['wd_scale_mult'],         'group':'scale'},
        {'params':pbld_p,    'lr':LR*p['pbld_lr_factor'],        'weight_decay':WD,                            'group':'pbld'},
        {'params':first_w_p, 'lr':LR*p['first_layer_lr_factor'], 'weight_decay':WD*p['first_layer_wd_factor'], 'group':'first_w'},
        {'params':other_w_p, 'lr':LR,                            'weight_decay':WD,                            'group':'other_w'},
        {'params':bias_p,    'lr':LR*p['lr_bias_mult'],          'weight_decay':WD*p['wd_bias_mult'],          'group':'bias'},
    ]

def smooth_ce_loss(y_true,y_pred,ls=0.0,class_weights=None):
    n_classes=y_pred.size(1)
    y_smooth=torch.full_like(y_pred,ls/n_classes)
    y_smooth.scatter_(1,y_true.unsqueeze(1),1.0-ls+ls/n_classes)
    loss=-(y_smooth*torch.log(y_pred.clamp(1e-15,1))).sum(dim=1)
    if class_weights is not None:
        w=class_weights[y_true]; return (loss*w).sum()/w.sum()
    return loss.mean()

print('Helpers defined ✅')

Helpers defined ✅


In [7]:
CONFIG = {
    'n_ens':           10,
    'embed_dim':       7,
    'onehot_thresh':   10,
    'hidden_dims':     [256, 512, 256],
    'dropout':         0.05,
    'p_drop_sched':    'expm4t',
    'activation':      nn.SiLU,
    'add_front_scale': True,
    'pbld_hidden_dim': 20,
    'pbld_out_dim':    5,
    'pbld_freq_scale': 3.0,
    'pbld_activation': nn.PReLU,
    'pbld_lr_factor':  0.093,
    'lr':                      0.01,
    'mom':                     0.9,
    'sq_mom':                  0.98,
    'lr_sched':                'flat_cos',
    'flat_ratio':              0.3,
    'first_layer_lr_factor':   1.0,
    'first_layer_wd_factor':   0.1,
    'lr_scale_mult':           10.0,
    'lr_bias_mult':            0.1,
    'weight_decay':            0.013,
    'wd_scale_mult':           0.1,
    'wd_bias_mult':            0.5,
    'grad_clip':               1.0,
    'ls_eps':       0.04,
    'ls_eps_sched': 'cos',
    'tfms': ['median_center', 'robust_scale'],
    'epochs':    15,
    'train_bs':  256,
    'eval_bs':   10240,
    'verbosity': 2,
    'use_early_stopping':                     True,
    'early_stopping_additive_patience':       3,
    'early_stopping_multiplicative_patience': 1,
    'device':       'cuda',
    'random_state': 42,
}

FOLDS = 5
SEED  = 42
TE    = True
print('Config set ✅')

Config set ✅


In [8]:
class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kwargs):
        self.params = {**CONFIG, **kwargs}

    def fit(self, X_train, y_train, X_val, y_val,
            cat_col_names=None, ckpt_path='realmlp_ckpt.pth', X_test=None):
        p   = self.params
        dev = torch.device(p['device'] if torch.cuda.is_available() else 'cpu')
        verbose = p['verbosity']
        cat_col_names = cat_col_names or []
        num_col_names = [c for c in X_train.columns if c not in cat_col_names]

        X_tr_num  = X_train[num_col_names].values.astype(np.float32)
        X_val_num = X_val[num_col_names].values.astype(np.float32)
        X_tr_cat  = X_train[cat_col_names].values.astype(np.int64)
        X_val_cat = X_val[cat_col_names].values.astype(np.int64)
        y_tr = np.asarray(y_train); y_v = np.asarray(y_val)

        self.preprocessor_ = NumericalPreprocessor(p['tfms'])
        self.preprocessor_.fit(X_tr_num)
        X_tr_num  = self.preprocessor_.transform(X_tr_num)
        X_val_num = self.preprocessor_.transform(X_val_num)

        self.cat_col_names_ = cat_col_names
        self.num_col_names_ = num_col_names
        all_cat = [X_tr_cat, X_val_cat]
        if X_test is not None:
            all_cat.append(X_test[cat_col_names].values.astype(np.int64))
        cat_dims = (np.concatenate(all_cat,axis=0).max(axis=0)+1).tolist() if cat_col_names else []
        self.cat_dims_ = cat_dims
        if cat_dims:
            cat_max = np.array(cat_dims)-1
            X_tr_cat  = np.clip(X_tr_cat,  0, cat_max)
            X_val_cat = np.clip(X_val_cat, 0, cat_max)

        classes = np.unique(y_tr); self.classes_ = classes
        weights_np = compute_class_weight('balanced',classes=classes,y=y_tr)
        class_weights = torch.as_tensor(weights_np,dtype=torch.float32,device=dev)

        n_classes   = len(classes)
        self.model_ = RealMLP(output_dim=n_classes,cat_dims=cat_dims,
                              n_numerical=X_tr_num.shape[1],cfg=p).to(dev)
        param_groups = get_parameter_groups(self.model_,p)
        for g in param_groups: g['lr_base']=g['lr']
        optimizer = torch.optim.AdamW(param_groups,betas=(p['mom'],p['sq_mom']))

        Xtn=torch.as_tensor(X_tr_num, dtype=torch.float32,device=dev)
        Xtc=torch.as_tensor(X_tr_cat, dtype=torch.long,   device=dev)
        ytt=torch.as_tensor(y_tr,     dtype=torch.long,   device=dev)
        Xvn=torch.as_tensor(X_val_num,dtype=torch.float32,device=dev)
        Xvc=torch.as_tensor(X_val_cat,dtype=torch.long,   device=dev)

        n_ens=p['n_ens']; total_steps=p['epochs']*len(y_tr)
        train_order=np.arange(len(y_tr))
        best_score,best_epoch,best_val_probs=-np.inf,0,None
        self.ckpt_path_=ckpt_path

        for epoch in range(p['epochs']):
            self.model_.train()
            for start in range(0,len(y_tr),p['train_bs']):
                progress=(epoch*len(y_tr)+start)/total_steps
                idx=train_order[start:start+p['train_bs']]
                for g in optimizer.param_groups:
                    g['lr']=apply_schedule(g['lr_base'],progress,p['lr_sched'],p['flat_ratio'])
                optimizer.zero_grad()
                y_pred=self.model_(Xtn[idx],Xtc[idx])
                ls_val  =apply_schedule(p['ls_eps'],  progress,p['ls_eps_sched'], p['flat_ratio'])
                drop_val=apply_schedule(p['dropout'], progress,p['p_drop_sched'], p['flat_ratio'])
                for dm in self.model_._dropout_modules: dm.p=drop_val
                loss=smooth_ce_loss(ytt[idx].repeat_interleave(n_ens),
                                    y_pred.reshape(-1,n_classes),
                                    ls=ls_val,class_weights=class_weights)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model_.parameters(),p['grad_clip'])
                optimizer.step()
            np.random.shuffle(train_order)

            self.model_.eval()
            with torch.no_grad():
                val_probs=np.concatenate([
                    self.model_(Xvn[s:s+p['eval_bs']],Xvc[s:s+p['eval_bs']])
                        .mean(dim=1).cpu().numpy()
                    for s in range(0,len(y_v),p['eval_bs'])])
            score=balanced_accuracy_score(y_v,np.argmax(val_probs,axis=1))
            improved=score>best_score
            if improved:
                best_score,best_epoch,best_val_probs=score,epoch+1,val_probs.copy()
                torch.save(self.model_.state_dict(),ckpt_path)
            if verbose>=2:
                print(f'  epoch {epoch+1}/{p["epochs"]}  score={score:.5f}  best={best_score:.5f}  '
                      f'ls={ls_val:.4f}  drop={drop_val:.4f}'+(' ✓' if improved else ''))
            if p['use_early_stopping']:
                patience=best_epoch*p['early_stopping_multiplicative_patience']+p['early_stopping_additive_patience']
                if (epoch+1)>patience:
                    if verbose>=1: print(f'  Early stop at epoch {epoch+1} (best={best_epoch})')
                    break

        self.model_.load_state_dict(torch.load(ckpt_path,map_location=dev))
        self.best_score_,self.best_val_probs_=best_score,best_val_probs
        self._dev=dev
        if verbose>=1: print(f'  → best: {best_score:.5f} (epoch {best_epoch})')
        return self

    def predict_proba(self,X):
        X_num=self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        X_cat=np.clip(X[self.cat_col_names_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(X_num,dtype=torch.float32,device=self._dev)
        Xc=torch.as_tensor(X_cat,dtype=torch.long,   device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([
                self.model_(Xn[s:s+self.params['eval_bs']],Xc[s:s+self.params['eval_bs']])
                    .mean(dim=1).cpu().numpy()
                for s in range(0,len(X_num),self.params['eval_bs'])])

print('RealMLP_TD_Classifier defined ✅')

RealMLP_TD_Classifier defined ✅


In [9]:
%%time
def metric(y_true,y_pred):
    return balanced_accuracy_score(y_true,np.argmax(y_pred,axis=1))

skf = StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
oof_mlp  = np.zeros((len(X),y.nunique()))
test_mlp = np.zeros((len(X_test),y.nunique()))

for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y),1):
    X_tr  = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    X_tst = X_test.copy()

    if TE:
        encoder=TargetEncoder(cv=FOLDS,smooth='auto',shuffle=True,random_state=SEED)
        tr_enc =encoder.fit_transform(X_tr[combo_names], y.iloc[tr_idx])
        val_enc=encoder.transform(X_val[combo_names])
        tst_enc=encoder.transform(X_tst[combo_names])
        te_names=[f'_{col}TE_cls{c}' for col in combo_names for c in range(y.nunique())]
        X_tr[te_names]=tr_enc; X_val[te_names]=val_enc; X_tst[te_names]=tst_enc

    print(f'\n{"#"*16}\n### Fold {fold}/{FOLDS}\n{"#"*16}')
    if fold==1: print(f'Total features: {len(X_tr.columns)}')

    model=RealMLP_TD_Classifier(**CONFIG)
    model.fit(X_tr,y.iloc[tr_idx],X_val,y.iloc[val_idx],
              cat_col_names=cat_cols,ckpt_path=f'model_fold{fold}.pth')
    oof_mlp[val_idx]=model.best_val_probs_
    test_mlp+=model.predict_proba(X_tst)/FOLDS

    print(f'Fold {fold} | Score: {metric(y.iloc[val_idx],oof_mlp[val_idx]):.5f}')
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f'\n{"="*30}')
print(f'RealMLP OOF: {metric(y,oof_mlp):.5f}')
print(f'{"="*30}')


################
### Fold 1/5
################
Total features: 105
  epoch 1/15  score=0.96555  best=0.96555  ls=0.0396  drop=0.0383 ✓
  epoch 2/15  score=0.96833  best=0.96833  ls=0.0383  drop=0.0293 ✓
  epoch 3/15  score=0.96868  best=0.96868  ls=0.0362  drop=0.0225 ✓
  epoch 4/15  score=0.96882  best=0.96882  ls=0.0334  drop=0.0172 ✓
  epoch 5/15  score=0.96873  best=0.96882  ls=0.0300  drop=0.0132
  epoch 6/15  score=0.96924  best=0.96924  ls=0.0262  drop=0.0101 ✓
  epoch 7/15  score=0.96849  best=0.96924  ls=0.0221  drop=0.0077
  epoch 8/15  score=0.96787  best=0.96924  ls=0.0179  drop=0.0059
  epoch 9/15  score=0.96872  best=0.96924  ls=0.0138  drop=0.0045
  epoch 10/15  score=0.96747  best=0.96924  ls=0.0100  drop=0.0035
  Early stop at epoch 10 (best=6)
  → best: 0.96924 (epoch 6)
Fold 1 | Score: 0.96924

################
### Fold 2/5
################
  epoch 1/15  score=0.96527  best=0.96527  ls=0.0396  drop=0.0383 ✓
  epoch 2/15  score=0.96674  best=0.96674  ls=0.0383  drop=

In [10]:
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

lgb_features = [
    c for c in X.columns
    if c not in {'spectral_type','galaxy_population'}
    and X[c].dtype != 'category'
]
print(f'LightGBM features: {len(lgb_features)}')

lgb_params = {
    'objective':'multiclass','num_class':3,'metric':'multi_logloss',
    'learning_rate':0.05,'num_leaves':127,'min_child_samples':50,
    'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':5,
    'reg_alpha':0.1,'reg_lambda':0.1,'is_unbalance':True,
    'verbose':-1,'n_jobs':-1,'seed':42,
}

oof_lgb  = np.zeros((len(X),3))
test_lgb = np.zeros((len(X_test),3))

for fold,(tr_idx,val_idx) in enumerate(skf.split(X,y),1):
    X_tr_l  = X.iloc[tr_idx][lgb_features].astype(float)
    X_val_l = X.iloc[val_idx][lgb_features].astype(float)
    dtrain  = lgb.Dataset(X_tr_l,label=y.iloc[tr_idx])
    dval    = lgb.Dataset(X_val_l,label=y.iloc[val_idx],reference=dtrain)
    model_lgb=lgb.train(lgb_params,dtrain,num_boost_round=1000,valid_sets=[dval],
                        callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(200)])
    val_prob=model_lgb.predict(X_val_l)
    oof_lgb[val_idx]=val_prob
    test_lgb+=model_lgb.predict(X_test[lgb_features].astype(float))/FOLDS
    score=balanced_accuracy_score(y.iloc[val_idx],np.argmax(val_prob,axis=1))
    print(f'Fold {fold} LGB | {score:.5f} | best_iter={model_lgb.best_iteration}')

print(f'\nLightGBM OOF: {balanced_accuracy_score(y,np.argmax(oof_lgb,axis=1)):.5f}')

LightGBM features: 67
[200]	valid_0's multi_logloss: 0.0881015
Fold 1 LGB | 0.95628 | best_iter=303
[200]	valid_0's multi_logloss: 0.0888516
Fold 2 LGB | 0.95781 | best_iter=254
[200]	valid_0's multi_logloss: 0.0903956
Fold 3 LGB | 0.95713 | best_iter=253
[200]	valid_0's multi_logloss: 0.0887593
Fold 4 LGB | 0.95685 | best_iter=286
[200]	valid_0's multi_logloss: 0.0888301
Fold 5 LGB | 0.95784 | best_iter=282

LightGBM OOF: 0.95718


In [11]:
print('Searching best blend weight...')
best_w,best_score=0.5,0
for w in np.arange(0.3,1.0,0.05):
    blend=w*oof_mlp+(1-w)*oof_lgb
    score=balanced_accuracy_score(y,np.argmax(blend,axis=1))
    print(f'  w_mlp={w:.2f} → {score:.5f}')
    if score>best_score: best_score,best_w=score,w

print(f'\n✅ Best: w_mlp={best_w:.2f}, w_lgb={1-best_w:.2f} → OOF={best_score:.5f}')
test_blend=best_w*test_mlp+(1-best_w)*test_lgb

pd.DataFrame(oof_mlp, columns=['GALAXY','QSO','STAR']).to_csv('oof_mlp.csv', index=False)
pd.DataFrame(oof_lgb, columns=['GALAXY','QSO','STAR']).to_csv('oof_lgb.csv', index=False)
pd.DataFrame(test_mlp,columns=['GALAXY','QSO','STAR']).to_csv('test_mlp.csv',index=False)
pd.DataFrame(test_lgb,columns=['GALAXY','QSO','STAR']).to_csv('test_lgb.csv',index=False)
np.save('oof_mlp.npy',  oof_mlp)
np.save('oof_lgb.npy',  oof_lgb)
np.save('test_mlp.npy', test_mlp)
np.save('test_lgb.npy', test_lgb)

print('✅ OOF and test probability files saved!')
print(f'   oof_mlp:{oof_mlp.shape} | test_mlp:{test_mlp.shape}')
print(f'   oof_lgb:{oof_lgb.shape} | test_lgb:{test_lgb.shape}')

label_map={0:'GALAXY',1:'QSO',2:'STAR'}
sub=pd.DataFrame({'id':test_id,
                  'class':pd.Series(np.argmax(test_blend,axis=1)).map(label_map)})
sub.to_csv('submission.csv',index=False)
print(f'\n✅ Submission saved!')
print(sub['class'].value_counts())
sub.head(5)

Searching best blend weight...
  w_mlp=0.30 → 0.96266
  w_mlp=0.35 → 0.96377
  w_mlp=0.40 → 0.96475
  w_mlp=0.45 → 0.96582
  w_mlp=0.50 → 0.96660
  w_mlp=0.55 → 0.96724
  w_mlp=0.60 → 0.96768
  w_mlp=0.65 → 0.96788
  w_mlp=0.70 → 0.96809
  w_mlp=0.75 → 0.96820
  w_mlp=0.80 → 0.96830
  w_mlp=0.85 → 0.96823
  w_mlp=0.90 → 0.96822
  w_mlp=0.95 → 0.96810

✅ Best: w_mlp=0.80, w_lgb=0.20 → OOF=0.96830
✅ OOF and test probability files saved!
   oof_mlp:(577347, 3) | test_mlp:(247435, 3)
   oof_lgb:(577347, 3) | test_lgb:(247435, 3)

✅ Submission saved!
class
GALAXY    157801
QSO        51137
STAR       38497
Name: count, dtype: int64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [12]:
%%time
from xgboost import XGBClassifier

PHYSICS_FEATURES = [
    'alpha','delta','u','g','r','i','z','redshift',
    'u_g','g_r','r_i','i_z','g_z','u_r','u_z','g_i','r_z',
    'mag_mean','mag_std','mag_range',
    'log_redshift','redshift_sq','redshift_zone',
    'sin_alpha','cos_alpha','sin_delta','cos_delta',
    'phys_x','phys_y','phys_z',
    'redshift_x_gr','redshift_x_ur',
    'locus_gr_pred','locus_distance','locus_dist_abs',
    'spectral_enc','pop_enc','spec_x_pop',
]
PHYSICS_FEATURES = [c for c in PHYSICS_FEATURES if c in X.columns]
print(f'XGB features: {len(PHYSICS_FEATURES)}')

xgb_params = dict(
    objective='multi:softprob',
    num_class=3,
    tree_method='hist',
    device='cuda',
    learning_rate=0.05,
    max_depth=8,
    n_estimators=3000,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    early_stopping_rounds=100,
    eval_metric='mlogloss',
)

oof_xgb  = np.zeros((len(X), 3))
test_xgb = np.zeros((len(X_test), 3))
xgb_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr  = X.iloc[tr_idx][PHYSICS_FEATURES].astype(np.float32)
    X_val = X.iloc[val_idx][PHYSICS_FEATURES].astype(np.float32)
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    classes = np.unique(y_tr)
    cw = compute_class_weight('balanced', classes=classes, y=y_tr)
    sw = np.array([dict(zip(classes, cw))[c] for c in y_tr])

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, sample_weight=sw,
              eval_set=[(X_val, y_val)], verbose=300)

    val_prob = model.predict_proba(X_val)
    oof_xgb[val_idx] = val_prob
    test_xgb += model.predict_proba(X_test[PHYSICS_FEATURES].astype(np.float32)) / FOLDS

    score = balanced_accuracy_score(y_val, np.argmax(val_prob, axis=1))
    xgb_scores.append(score)
    print(f'Fold {fold} | BAC={score:.5f} | best_iter={model.best_iteration}')

print(f'\nMean: {np.mean(xgb_scores):.5f} ± {np.std(xgb_scores):.5f}')
print(f'OOF BAC: {balanced_accuracy_score(y, np.argmax(oof_xgb,axis=1)):.5f}')


XGB features: 38
[0]	validation_0-mlogloss:1.03515
[300]	validation_0-mlogloss:0.10531
[600]	validation_0-mlogloss:0.09811
[900]	validation_0-mlogloss:0.09525
[1200]	validation_0-mlogloss:0.09443
[1341]	validation_0-mlogloss:0.09449
Fold 1 | BAC=0.96436 | best_iter=1241
[0]	validation_0-mlogloss:1.03527
[300]	validation_0-mlogloss:0.10888
[600]	validation_0-mlogloss:0.10096
[900]	validation_0-mlogloss:0.09800
[1200]	validation_0-mlogloss:0.09703
[1295]	validation_0-mlogloss:0.09707
Fold 2 | BAC=0.96437 | best_iter=1195
[0]	validation_0-mlogloss:1.03541
[300]	validation_0-mlogloss:0.11078
[600]	validation_0-mlogloss:0.10275
[900]	validation_0-mlogloss:0.09973
[1200]	validation_0-mlogloss:0.09894
[1357]	validation_0-mlogloss:0.09897
Fold 3 | BAC=0.96334 | best_iter=1257
[0]	validation_0-mlogloss:1.03527
[300]	validation_0-mlogloss:0.10759
[600]	validation_0-mlogloss:0.10019
[900]	validation_0-mlogloss:0.09747
[1200]	validation_0-mlogloss:0.09680
[1386]	validation_0-mlogloss:0.09692
Fold 

In [13]:
pd.DataFrame(oof_xgb,  columns=['GALAXY','QSO','STAR']).to_csv('oof_xgb.csv', index=False)
pd.DataFrame(test_xgb, columns=['GALAXY','QSO','STAR']).to_csv('test_xgb.csv', index=False)
np.save('oof_xgb.npy', oof_xgb)
np.save('test_xgb.npy', test_xgb)

label_map = {0:'GALAXY',1:'QSO',2:'STAR'}
sub_xgb = pd.DataFrame({'id': test_id, 'class': pd.Series(np.argmax(test_xgb,axis=1)).map(label_map)})
sub_xgb.to_csv('submission_xgb.csv', index=False)

print('✅ Saved oof_xgb.csv, test_xgb.csv (+npy), submission_xgb.csv')
print(sub_xgb['class'].value_counts())


✅ Saved oof_xgb.csv, test_xgb.csv (+npy), submission_xgb.csv
class
GALAXY    159227
QSO        50732
STAR       37476
Name: count, dtype: int64
